# Praktikum 3
Klasifikasi Naive Bayes dengan Data Multimoda (Teks SMS)

**Nilai multinomial** adalah nilai yang diperoleh dari proses **menghitung** (counting) — contoh klasiknya adalah jumlah kemunculan kata pada klasifikasi teks.

**Dataset:** `spam.csv` — SMS Spam Collection (5.572 pesan) dengan label `spam` (SMS sampah) dan `ham` (SMS normal).

### Langkah 1 - Load Data
Perhatikan `encoding='latin-1'` karena berkas tidak memakai UTF-8.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('spam.csv', encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


### Langkah 2 - Preprocessing
Tahapan: (a) buang kolom tak terpakai, (b) ganti nama kolom, (c) inspeksi, (d) encode label, (e) pisahkan fitur dan label.

#### Langkah 2a - Drop Kolom
Hanya dua kolom pertama (`v1` = label, `v2` = teks SMS) yang dipakai.

In [2]:
# Drop 3 kolom terakhir
# df.drop(df.iloc[:, 2:], axis=1) pada modul diganti bentuk yang lebih eksplisit
df = df.drop(df.columns[2:], axis=1)

# Cek data
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


#### Langkah 2b - Rename Kolom & Inspeksi

In [3]:
# Ubah nama kolom agar mudah dibaca
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

# Cek jumlah data per kelas
print(df['Labels'].value_counts())
print('\n')

# Cek kelengkapan data
print(df.info())
print('\n')

# Cek statistik deskriptif
print(df.describe())

Labels
ham     4825
spam     747
Name: count, dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Labels  5572 non-null   str  
 1   SMS     5572 non-null   str  
dtypes: str(2)
memory usage: 541.4 KB
None


       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


#### Langkah 2c - Encoding Label
`spam` → 1, `ham` → 0.

In [4]:
# Data untuk label
new_labels = {'spam': 1, 'ham': 0}

df['Labels'] = df['Labels'].map(new_labels)
df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


#### Langkah 2d - Pisahkan Fitur dengan Label

> **Penyesuaian (pandas 3.0):** kolom teks kini ber-*backing* Arrow, sehingga `.values` menghasilkan `ArrowStringArray` yang membuat `train_test_split` gagal. Dipakai `.to_numpy(dtype=object)`.

In [5]:
X = df['SMS'].to_numpy(dtype=object)
y = df['Labels'].to_numpy()

### Langkah 3 - Ekstraksi Fitur (Bag of Words)
Gunakan `CountVectorizer`. **Penting:** split data **dulu**, baru *fit* vectorizer pada data latih saja, agar tidak terjadi *information leakage* dari data uji.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Split data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

# Inisiasi CountVectorizer
bow = CountVectorizer()

# Fit + transform data latih
X_train = bow.fit_transform(X_train)

# Data uji hanya di-transform (bukan fit) agar tetap "asing" bagi model
X_test = bow.transform(X_test)

> **Penyesuaian:** `bow.get_feature_names()` sudah dihapus pada scikit-learn ≥ 1.2, diganti `get_feature_names_out()`.

In [7]:
print(len(bow.get_feature_names_out()))
print(f'Dimensi data: {X_train.shape}')

7727
Dimensi data: (4457, 7727)


### Langkah 4 - Training dan Evaluasi Model
Gunakan **MultinomialNB**, algoritma yang tepat untuk fitur cacah kata.

In [8]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

mnb = MultinomialNB()
mnb.fit(X_train, y_train)

acc_train = accuracy_score(y_train, mnb.predict(X_train))
acc_test = accuracy_score(y_test, mnb.predict(X_test))

print(f'Hasil akurasi data train: {acc_train}')
print(f'Hasil akurasi data test: {acc_test}')

Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9775784753363229


### Langkah 5 - Analisis Hasil
Lihat confusion matrix dan contoh kesalahan prediksi untuk memahami perilaku model.

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = mnb.predict(X_test)
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nLaporan Klasifikasi:\n', classification_report(y_test, y_pred))

Confusion Matrix:
 [[951   3]
 [ 22 139]]

Laporan Klasifikasi:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       954
           1       0.98      0.86      0.92       161

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.95      1115
weighted avg       0.98      0.98      0.98      1115



### Kesimpulan

* **MultinomialNB** dengan fitur Bag-of-Words memberi akurasi sangat tinggi (**≈ 0,99 latih / ≈ 0,98 uji**) — Naive Bayes memang sangat efektif untuk klasifikasi teks.
* Selisih akurasi latih dan uji kecil → model tidak overfit berlebihan meski dimensi fitur mencapai ribuan kata.
* Asumsi independensi antar kata memang tidak realistis secara linguistik, tetapi tetap bekerja baik pada tugas klasifikasi teks.
* Kunci praktik yang benar: `fit_transform` hanya pada data latih, `transform` pada data uji.